In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CELL 2 — Chargement
# ============================================================
df = pd.read_csv("../data/processed/clean_data.csv")
print(df.shape)
df.head()

In [ ]:
# ============================================================
# CELL 3 — Feature Engineering CORRIGÉ
# ============================================================
# Supprimer outliers extrêmes — garder entre 1% et 99%
Q99 = df['prix'].quantile(0.99)
Q01 = df['prix'].quantile(0.01)
df = df[(df['prix'] >= Q01) & (df['prix'] <= Q99)]
print(f"Dataset après nettoyage : {df.shape[0]} annonces")
print(f"Prix min : {df['prix'].min():,.0f} DH")
print(f"Prix max : {df['prix'].max():,.0f} DH")
df['ville'] = df['localisation'].str.split(',').str[0].str.strip().str.lower()

ville_counts = df['ville'].value_counts()
villes_valides = ville_counts[ville_counts >= 20].index
df = df[df['ville'].isin(villes_valides)]

# Features numériques — pas de leakage ici
df['surface_log'] = np.log1p(df['surface'])
df['surface_carree'] = df['surface'] ** 2
df['ratio_chambres_surface'] = df['chambres'] / df['surface']
df['ratio_sdb_surface'] = df['salles_de_bain'] / df['surface']
df['total_pieces'] = df['chambres'] + df['salles_de_bain']
df['est_agence'] = (df['type_vendeur'] == 'STORE').astype(int)

In [ ]:
# ============================================================
# CELL 4 — Features SANS prix_median_ville
# ============================================================
features = [
    'surface',
    'surface_log',
    'chambres',
    'salles_de_bain',
    'etage',
    'total_pieces',
    'ratio_chambres_surface',
    'est_agence',
]

# Encoder la ville proprement
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['ville_encoded'] = le.fit_transform(df['ville'])
features.append('ville_encoded')

X = df[features].copy()
y = df['prix'].copy()

X = X.dropna()
y = y[X.index]

In [ ]:
# ============================================================
# CELL 5 — Split stratifié
# ============================================================
df_model = df.loc[X.index].copy()
prix_bins = pd.qcut(y, q=5, labels=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=prix_bins  # ← garantit distribution égale
)

# Calculer prix_median_ville sur train uniquement
train_df = df.loc[X_train.index].copy()
prix_median_par_ville = train_df.groupby('ville')['prix'].median()

X_train = X_train.copy()
X_test = X_test.copy()

X_train['prix_median_ville'] = df.loc[X_train.index, 'ville'].map(prix_median_par_ville)
X_test['prix_median_ville'] = df.loc[X_test.index, 'ville'].map(prix_median_par_ville)

mediane_globale = y_train.median()
X_train['prix_median_ville'] = X_train['prix_median_ville'].fillna(mediane_globale)
X_test['prix_median_ville'] = X_test['prix_median_ville'].fillna(mediane_globale)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Max prix train: {y_train.max():,.0f} DH")
print(f"Max prix test:  {y_test.max():,.0f} DH")

In [ ]:
# ============================================================
# CELL 6 — Modèle : GradientBoosting (meilleur que RF ici)
# ============================================================
# Pourquoi GBR plutôt que RF ?
# - Dataset petit (~3000 lignes) : GBR performe mieux
# - RF sur petits datasets tend à overfit
# - GBR est moins sensible aux features bruitées

model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=10,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train_log)  # ← y_train_log ici

In [ ]:
# ============================================================
# CELL 7 — Évaluation CORRECTE
# ============================================================
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)
r2_log = r2_score(y_test_log, y_pred_log)

print(f"MAE          : {mae:,.0f}")
print(f"RMSE         : {rmse:,.0f}")
print(f"R² réel      : {r2:.4f}")
print(f"R² log       : {r2_log:.4f}")

cv_scores = cross_val_score(model, X_train, y_train_log, cv=5, scoring='r2')
print(f"\nCV R² moyen  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ============================================================
# CELL 8 — Importance des features
# ============================================================
feature_names = X_train.columns.tolist()  # ← prend les vrais noms depuis X_train

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.title("Importance des features — GradientBoosting")
plt.tight_layout()
plt.show()

print(importance_df.sort_values('Importance', ascending=False))

In [ ]:
# ============================================================
# CELL 9 — Analyse des résidus (diagnostic qualité)
# ============================================================
residus = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residus, alpha=0.4)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Prédictions")
axes[0].set_ylabel("Résidus")
axes[0].set_title("Résidus vs Prédictions")

axes[1].scatter(y_test, y_pred, alpha=0.4)
axes[1].plot([y_test.min(), y_test.max()], 
             [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel("Prix réels")
axes[1].set_ylabel("Prix prédits")
axes[1].set_title("Réel vs Prédit")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 10 — Sauvegarde
# ============================================================
import joblib

joblib.dump(model, '../models/model_prix_v2.pkl')
joblib.dump(X_train.columns.tolist(), '../models/features_v2.pkl')
joblib.dump(prix_median_par_ville, '../models/prix_median_ville.pkl')

print(" Modèle sauvegardé")
print("Features:", X_train.columns.tolist())

In [ ]:
# ============================================================
# CELL 11 — Test avec une annonce fictive
# ============================================================
feature_order = X_train.columns.tolist()

test_annonce = pd.DataFrame([{
    'surface': 80,
    'surface_log': np.log1p(80),
    'chambres': 2,
    'salles_de_bain': 1,
    'etage': 2,
    'total_pieces': 3,
    'ratio_chambres_surface': 2/80,
    'ville_encoded': le.transform(['casablanca'])[0],
    'est_agence': 0,
    'prix_median_ville': prix_median_par_ville.get('casablanca', y_train.median())
}])

test_annonce = test_annonce[feature_order]

pred_log = model.predict(test_annonce)[0]
pred_prix = np.expm1(pred_log)

print("=== Annonce test ===")
print(f"Ville          : Casablanca")
print(f"Surface        : 80 m²")
print(f"Chambres       : 2")
print(f"Salle de bain  : 1")
print(f"Étage          : 2")
print(f"\nPrix prédit    : {pred_prix:,.0f} DH")

In [ ]:
# ============================================================
# CELL 12 — Test par ville
# ============================================================
feature_order = X_train.columns.tolist()
villes_test = ['casablanca', 'rabat', 'marrakech', 'tanger', 'agadir']

print("Appartement 80m², 2 chambres, 1 sdb, étage 2 :\n")
for ville in villes_test:
    annonce = pd.DataFrame([{
        'surface': 80,
        'surface_log': np.log1p(80),
        'chambres': 2,
        'salles_de_bain': 1,
        'etage': 2,
        'total_pieces': 3,
        'ratio_chambres_surface': 2/80,
        'ville_encoded': le.transform([ville])[0],
        'est_agence': 0,
        'prix_median_ville': prix_median_par_ville.get(ville, y_train.median())
    }])
    annonce = annonce[feature_order]
    pred = np.expm1(model.predict(annonce)[0])
    print(f"{ville.capitalize():<15} → {pred:,.0f} DH")

In [ ]:
# ============================================================
# CELL 13 — Comparaison réel vs prédit sur 10 annonces
# ============================================================
sample = X_test.sample(10, random_state=1)
y_sample_real = y_test[sample.index]
y_sample_pred = np.expm1(model.predict(sample))

comparaison = pd.DataFrame({
    'Prix réel (DH)' : y_sample_real.values,
    'Prix prédit (DH)': y_sample_pred.round(0),
    'Erreur (DH)'    : abs(y_sample_real.values - y_sample_pred).round(0),
    'Erreur (%)'     : (abs(y_sample_real.values - y_sample_pred) / y_sample_real.values * 100).round(1)
})

print(comparaison.to_string(index=False))
print(f"\nErreur moyenne : {comparaison['Erreur (%)'].mean():.1f}%")
print(f"Erreur médiane : {comparaison['Erreur (%)'].median():.1f}%")

In [ ]:
# DIAGNOSTIC COMPLET
print("=== Distribution du prix ===")
print(y.describe())
print(f"\nPrix > 5M DH : {(y > 5_000_000).sum()} annonces")
print(f"Prix > 10M DH : {(y > 10_000_000).sum()} annonces")

print("\n=== Distribution y_train_log ===")
print(y_train_log.describe())

print("\n=== Distribution y_test_log ===")
print(y_test_log.describe())

# Visualiser
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
y.hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution prix brut")
y_train_log.hist(bins=50, ax=axes[1])
axes[1].set_title("Distribution log(prix)")
plt.show()